# Notebook 02 — GraphRAG Retriever Development

Builds sentence-transformer embeddings for all IFC graph nodes, validates seed retrieval, compares Flat RAG vs GraphRAG subgraph quality, and computes GGS.

**No GPU required** (embedding on CPU is acceptable for the IFC graph size).

In [ ]:
import os, sys, json
REPO = 'ifc-graphrag-dt'
if os.path.exists(REPO): !cd {REPO} && git pull --quiet
else: !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
sys.path.insert(0, REPO)

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.graph_embedder import GraphEmbedder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal
from evaluation.metrics.ggs import GGSScorer
from benchmark.dtah_bench import DTAHBench

IFC_PATH    = f'{REPO}/benchmark/ifc_reference_models/duplex.ifc'
GRAPH_CACHE = f'{REPO}/outputs/graphs/ifc_graph.json'
EMB_CACHE   = f'{REPO}/outputs/embedders/graph_embedder'
os.makedirs(f'{REPO}/outputs/embedders', exist_ok=True)
print('Imports OK')

In [ ]:
# ── Cell 2: Load graph ────────────────────────────────────────────────────────
if Path(GRAPH_CACHE).exists():
    G = IFCGraphBuilder.load_graph(GRAPH_CACHE)
else:
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_CACHE)

print(f'Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# ── Cell 3: Build or load embedder ───────────────────────────────────────────
if Path(EMB_CACHE).exists():
    print('Loading cached embedder...')
    embedder = GraphEmbedder.load(EMB_CACHE)
else:
    print('Building embeddings (this takes ~2-5 minutes on CPU)...')
    embedder = GraphEmbedder(model_name='sentence-transformers/all-MiniLM-L6-v2')
    embedder.fit(G)
    embedder.save(EMB_CACHE)

print(f'Embedder ready: {len(embedder._node_ids)} nodes indexed')

In [ ]:
# ── Cell 4: Seed retrieval quality test ──────────────────────────────────────
TEST_PROMPTS = [
    ('T1-MEP-001', 'Generate a centrifugal pump', ['IfcPump', 'IfcFlowMovingDevice']),
    ('T2-MEP-001', 'Generate a centrifugal pump connected to two pipe segments', ['IfcPump', 'IfcPipeSegment']),
    ('T2-MEP-002', 'Generate a globe valve mounted on a horizontal pipe segment', ['IfcValve', 'IfcPipeSegment']),
    ('T3-HVAC-001', 'Generate a central AHU system serving four VAV zones', ['IfcAirToAirHeatRecovery', 'IfcAirTerminalBox']),
]

print(f'{"Prompt ID":<15} {"Prompt (truncated)":<52} {"Top-5 retrieved types"}')
print('-' * 110)
for pid, prompt, expected in TEST_PROMPTS:
    seeds = embedder.retrieve_seeds(prompt, top_k=5)
    retrieved_types = [s['ifc_type'] for s in seeds]
    hit = any(e in retrieved_types for e in expected)
    flag = '✓' if hit else '⚠'
    print(f'{flag} {pid:<13} {prompt[:50]:<52} {retrieved_types}')

In [ ]:
# ── Cell 5: Flat RAG vs GraphRAG subgraph comparison ─────────────────────────
# This is the core experimental comparison that justifies the paper

test_prompt = 'Generate a pump room with two pumps in parallel, inlet and outlet headers, and isolation valves'
print(f'Test prompt: {test_prompt}\n')

# Flat RAG: top-k seeds ONLY, no traversal
flat_seeds = embedder.retrieve_seeds(test_prompt, top_k=5)
flat_ids   = [s['node_id'] for s in flat_seeds]
flat_subgraph = G.subgraph(flat_ids).copy()  # induced — no traversal

# GraphRAG: k-hop traversal from same seeds
traversal = KHopTraversal(G, max_depth=3, bidirectional=True)
graph_result = traversal.traverse(flat_ids)
graph_subgraph = graph_result.subgraph

print('COMPARISON: Flat RAG vs GraphRAG')
print(f'{"Metric":<30} {"Flat RAG":>12} {"GraphRAG":>12}')
print('-' * 56)
print(f'{"Nodes recovered":<30} {flat_subgraph.number_of_nodes():>12} {graph_subgraph.number_of_nodes():>12}')
print(f'{"Edges recovered":<30} {flat_subgraph.number_of_edges():>12} {graph_subgraph.number_of_edges():>12}')

flat_rel_types  = set(d.get('relation_type','') for _,_,d in flat_subgraph.edges(data=True))
graph_rel_types = set(d.get('relation_type','') for _,_,d in graph_subgraph.edges(data=True))
print(f'{"Relation types found":<30} {len(flat_rel_types):>12} {len(graph_rel_types):>12}')
print(f'\nFlat RAG relation types: {flat_rel_types or "NONE — edges lost"}')
print(f'GraphRAG relation types: {graph_rel_types}')

In [ ]:
# ── Cell 6: GGS computation on pilot prompts ─────────────────────────────────
bench = DTAHBench(pilot_mode=True)
ggs_scorer = GGSScorer()

results = []
for tier in [1, 2, 3]:
    prompts = bench.load_tier(tier)
    for p in prompts[:5]:  # sample 5 per tier for this notebook
        seeds = embedder.retrieve_seeds(p['prompt'], top_k=5)
        seed_ids = [s['node_id'] for s in seeds]

        # Flat RAG subgraph
        flat_sg = G.subgraph(seed_ids).copy()

        # GraphRAG subgraph (use tier-appropriate depth)
        depth = {1: 1, 2: 2, 3: 4}[tier]
        trav  = KHopTraversal(G, max_depth=depth, bidirectional=True)
        graph_sg = trav.traverse(seed_ids).subgraph

        # Use graph_sg as proxy ground truth for now (will be replaced by annotated GT)
        # This measures how much MORE GraphRAG recovers vs Flat RAG
        ggs_flat  = ggs_scorer.score(flat_sg,  graph_sg, prompt_id=f'{p["id"]}_flat')
        ggs_graph = ggs_scorer.score(graph_sg, graph_sg, prompt_id=f'{p["id"]}_graph')

        results.append({'id': p['id'], 'tier': tier, 'prompt': p['prompt'][:50],
                        'flat_ggs': ggs_flat.total, 'graph_ggs': ggs_graph.total,
                        'flat_edge_recall': ggs_flat.edge_recall,
                        'graph_edge_recall': ggs_graph.edge_recall})

print(f'{"ID":<15} {"T":>2} {"Flat GGS":>10} {"Graph GGS":>10} {"Flat Edge R":>12} {"Prompt (trunc)"}')
print('-' * 90)
for r in results:
    print(f'{r["id"]:<15} {r["tier"]:>2} {r["flat_ggs"]:>10.3f} {r["graph_ggs"]:>10.3f} '
          f'{r["flat_edge_recall"]:>12.3f} {r["prompt"]}')

In [ ]:
# ── Cell 7: GGS by tier bar chart ────────────────────────────────────────────
import numpy as np

tiers = [1, 2, 3]
flat_by_tier  = [[r['flat_ggs']  for r in results if r['tier']==t] for t in tiers]
graph_by_tier = [[r['graph_ggs'] for r in results if r['tier']==t] for t in tiers]

flat_means  = [np.mean(v) if v else 0 for v in flat_by_tier]
graph_means = [np.mean(v) if v else 0 for v in graph_by_tier]

x = np.arange(len(tiers))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, flat_means,  w, label='Flat RAG',  color='#888780')
ax.bar(x + w/2, graph_means, w, label='GraphRAG',  color='#2E5FA3')
ax.set_xticks(x)
ax.set_xticklabels(['Tier 1\n(Asset)', 'Tier 2\n(Assembly)', 'Tier 3\n(System)'])
ax.set_ylabel('GGS (Graph Grounding Score)')
ax.set_ylim(0, 1.05)
ax.set_title('Flat RAG vs GraphRAG — GGS by Tier (proxy GT)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{REPO}/outputs/figures/02_ggs_comparison.png', dpi=150)
plt.show()
print('Retriever development complete. Proceed to Notebook 03.')